In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_producto")

In [0]:
spark.sql(
    """
    create table if not exists workspace.gold.dim_producto(
        id_producto bigint,
        codigo_producto string,
        descripcion string,
        precio_venta decimal(12,2),
        costo_venta decimal(12,2),
        almacen bigint
    )
   
    """
)

In [0]:
from pyspark.sql import functions as F
df_dim_producto=spark.table("workspace.silver.tbl_productos")
df_dim_producto=(df_dim_producto
                      .dropDuplicates(["codigo_producto"])
                      .withColumn("id_producto",F.monotonically_increasing_id()+1)
                      .select("id_producto","codigo_producto","descripcion","precio_venta","costo_venta","almacen")
                      )



In [0]:
df_dim_producto.write.format("delta")\
    .option("mergeShema","true")\
    .mode("overwrite")\
    .saveAsTable("workspace.gold.dim_producto")

In [0]:
%sql
select *from workspace.gold.dim_producto